# Deliverable 4 — Portfolio Backtesting: 2×2 Experiment Grid

This notebook extends the single-stock analysis to the full Nasdaq-100 universe of 101 stocks.
Rather than allocating entirely to one stock, the system generates a weight vector across all
tickers each day, with weights summing to at most 1 and no short selling permitted.

**Two signals** are used to select stocks each day:
- **Momentum Signal**: Rank all 101 stocks by 20-day trailing return; select the top 25% (~25 stocks).
- **SMA Crossover Signal**: Select stocks where the 20-day SMA exceeds the 50-day SMA.

**Two weighting schemes** are applied to each selected set:
- **Equal Weight**: Each selected stock receives weight 1/N.
- **Inverse-Volatility Weight**: Weight proportional to 1/(20-day rolling vol), normalised to sum to 1.

This produces a 2×2 grid of four experiments, each run frictionless.

In [ ]:
%matplotlib inline
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

from backtester import load_prices, run_backtest
from backtester.metrics import format_metrics
from strategies.momentum import MomentumPortfolio
from strategies.sma_crossover import SMAPortfolio

In [ ]:
prices = load_prices('data/nasdaq100_daily_5y.csv')
print(f'Loaded: {prices.shape[0]} trading days × {prices.shape[1]} tickers')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')

In [ ]:
# 2×2 experiment grid — frictionless
specs = [
    ('Momentum Top 25% — Equal Wt',
     lambda: MomentumPortfolio(lookback=20, top_pct=0.25, weighting='equal')),
    ('Momentum Top 25% — Inv-Vol Wt',
     lambda: MomentumPortfolio(lookback=20, top_pct=0.25, weighting='inverse_vol')),
    ('SMA Crossover (20/50d) — Equal Wt',
     lambda: SMAPortfolio(short_window=20, long_window=50, weighting='equal')),
    ('SMA Crossover (20/50d) — Inv-Vol Wt',
     lambda: SMAPortfolio(short_window=20, long_window=50, weighting='inverse_vol')),
]

results = []
for label, factory in specs:
    r = run_backtest(prices, factory(), transaction_cost=0.0, name_override=label)
    results.append(r)

## Combined NAV Comparison

In [ ]:
colors = ['steelblue', 'dodgerblue', 'darkorange', 'sandybrown']
fig, ax = plt.subplots(figsize=(13, 6))
for r, c in zip(results, colors):
    nav = r['nav_series']
    ax.plot(nav.index, nav.values, label=r['strategy_name'], color=c, linewidth=1.5)
ax.set_title('Portfolio Backtesting — 2×2 Signal × Weighting Grid (frictionless)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('NAV (normalized to 1.0)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=9, frameon=True)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Metrics Table

In [ ]:
rows = []
for r in results:
    m = r['metrics']
    rows.append({
        'Strategy':        r['strategy_name'],
        'Cum. Return':     f"{m['cumulative_return']:+.2%}",
        'Ann. Volatility': f"{m['annualized_volatility']:.2%}",
        'Sharpe Ratio':    f"{m['sharpe_ratio']:.2f}",
        'Max Drawdown':    f"{-m['max_drawdown']:+.2%}",
        'Win Rate':        f"{m['win_rate']:.2%}",
    })
pd.DataFrame(rows).set_index('Strategy')

## Interpretation: Signal Quality

**Momentum dominates SMA crossover across both weighting schemes.**

The momentum signal (rank by 20-day return, take top 25%) achieves Sharpe 0.89 with equal
weighting versus 0.66 for SMA crossover. This reflects a fundamental difference in signal
responsiveness: a 20-day return ranking updates daily and reacts quickly to price acceleration.
A 20/50-day MA crossover is a lagging indicator by construction — the short average must
meaningfully diverge from the long average before a stock is selected, meaning entry occurs
well after the trend has started.

Both signals still produce positive Sharpe ratios, confirming that cross-sectional momentum
is a genuine and persistent effect in the Nasdaq-100 over this period.

## Interpretation: Inverse-Volatility vs Equal Weighting

**Inverse-volatility weighting consistently reduces both volatility AND returns, resulting
in lower Sharpe ratios on this dataset despite lower drawdowns.**

| | Momentum Signal | SMA Signal |
|---|---|---|
| Equal Wt Sharpe | **0.89** | **0.66** |
| Inv-Vol Wt Sharpe | 0.77 | 0.60 |
| Inv-Vol Drawdown advantage | -31.3% vs -36.6% | -33.1% vs -37.9% |

The explanation lies in the dataset's structure. During 2021–2026, the Nasdaq-100 was
dominated by high-volatility growth stocks — NVDA (+1300%), META, AMD — which were
simultaneously the highest-momentum and highest-volatility names. Inverse-volatility
weighting systematically *down-weights* these stocks and *up-weights* lower-volatility
defensive names (utilities, consumer staples).

The result: annual volatility drops from 23% to 20%, but cumulative return falls from
+143% to +96%. The return reduction outpaces the volatility reduction in proportional
terms, suppressing the Sharpe ratio.

**Takeaway:** Inverse-volatility weighting is well-suited to regimes where high-vol stocks
do not proportionally outperform low-vol stocks. In a strong tech-driven bull market,
it reduces drawdown but sacrifices return-per-unit-risk.

## Individual NAV Curves

In [ ]:
colors = ['steelblue', 'dodgerblue', 'darkorange', 'sandybrown']
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
axes = axes.flatten()
for ax, r, c in zip(axes, results, colors):
    nav = r['nav_series']
    ax.plot(nav.index, nav.values, color=c, linewidth=1.5)
    ax.set_title(r['strategy_name'], fontsize=10, fontweight='bold')
    ax.set_ylabel('NAV')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax.grid(True, alpha=0.3)
    m = r['metrics']
    ax.annotate(
        f"Sharpe: {m['sharpe_ratio']:.2f}  |  Max DD: {-m['max_drawdown']:+.1%}",
        xy=(0.02, 0.05), xycoords='axes fraction', fontsize=8
    )
fig.autofmt_xdate()
plt.suptitle('Portfolio Strategies — Individual NAV Curves', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()